In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Additional utilities
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")


In [ ]:
# Load stock price data
def load_stock_data(symbol='AAPL', period='2y'):
    """
    Load stock price data using yfinance
    
    Args:
        symbol: Stock symbol (e.g., 'AAPL', 'GOOGL')
        period: Time period ('1y', '2y', '5y', etc.)
        
    Returns:
        DataFrame with stock data
    """
    try:
        stock = yf.Ticker(symbol)
        data = stock.history(period=period)
        return data
    except Exception as e:
        print(f"Error loading data: {e}")
        # Create synthetic data as fallback
        dates = pd.date_range(start='2022-01-01', end='2024-01-01', freq='D')
        np.random.seed(42)
        price = 100 + np.cumsum(np.random.randn(len(dates)) * 0.5)
        return pd.DataFrame({'Close': price}, index=dates)

# Load and preprocess data
print("Loading stock price data...")
stock_data = load_stock_data('AAPL', '2y')

# Display basic information
print(f"Data shape: {stock_data.shape}")
print(f"Date range: {stock_data.index[0]} to {stock_data.index[-1]}")
print("\nFirst few rows:")
print(stock_data[['Close', 'Volume']].head())

# Visualize the data
plt.figure(figsize=(12, 6))
plt.plot(stock_data.index, stock_data['Close'])
plt.title('Stock Price Over Time')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Focus on closing price for prediction
data = stock_data['Close'].values.reshape(-1, 1)
print(f"\nClosing price data shape: {data.shape}")


In [ ]:
# Data preprocessing for time series
def create_sequences(data, window_size=60):
    """
    Create sequences for time series prediction
    
    Args:
        data: Time series data
        window_size: Number of time steps to look back
        
    Returns:
        X (sequences), y (targets)
    """
    X, y = [], []
    
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i, 0])
        y.append(data[i, 0])
    
    return np.array(X), np.array(y)

# Scale the data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

print("Data scaling completed")
print(f"Original data range: {data.min():.2f} to {data.max():.2f}")
print(f"Scaled data range: {scaled_data.min():.2f} to {scaled_data.max():.2f}")

# Create sequences
window_size = 60  # Use 60 days to predict the next day
X, y = create_sequences(scaled_data, window_size)

print(f"\nSequence creation completed")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Split data into train/validation/test sets
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.2)

X_train = X[:train_size]
y_train = y[:train_size]
X_val = X[train_size:train_size+val_size]
y_val = y[train_size:train_size+val_size]
X_test = X[train_size+val_size:]
y_test = y[train_size+val_size:]

print(f"\nData split completed:")
print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

# Reshape for RNN input (samples, time steps, features)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print(f"\nReshaped for RNN:")
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")
